# Model Context Protocol (MCP): A Complete Guide

## Overview

**Model Context Protocol (MCP)** is an open standard introduced by Anthropic in November 2024 that defines a universal, language-agnostic protocol for connecting AI agents to tools, data sources, and services. It is the USB standard for AI tool connectivity: one protocol, any agent, any tool.

Before MCP, every agent framework (LangChain, AutoGen, Semantic Kernel, Copilot Studio) had its own proprietary tool interface. A tool built for LangChain could not be used by a Copilot agent without rewriting it. MCP ends this fragmentation.

## What This Guide Covers

1. Core concepts: what MCP is, the server/client/host architecture
2. MCP vs LangChain `@tool` — the structural equivalence
3. Building an MCP server from scratch (finance + healthcare domain)
4. Converting existing LangChain agents to MCP servers
5. Building an MCP client that discovers and calls tools dynamically
6. Connecting LangChain agents to MCP servers
7. Multi-server composition — one agent, multiple MCP servers
8. Production patterns: auth, error handling, schema validation

## Prerequisites

- Python 3.10+
- Azure OpenAI resource with a GPT-4o deployment
- `.env` file with credentials (Cell 2 creates a template)

In [1]:
%pip install langchain langchain-openai langgraph mcp openai python-dotenv --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-ada-002
AZURE_SEARCH_ENDPOINT=https://<TBD>.search.windows.net
AZURE_SEARCH_KEY=<TBD>
AZURE_SEARCH_INDEX=enterprise-rag-data
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
/content/.env


In [3]:

import os, json, asyncio, uuid
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

llm = AzureChatOpenAI(
    azure_deployment=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    temperature=0,
    max_tokens=4096,
)
print('Setup complete. LLM:', type(llm).__name__)

Setup complete. LLM: AzureChatOpenAI


## Section 1 — Core Concepts

### The Three Roles

MCP defines three roles that map cleanly to real systems:

```
┌─────────────────────────────────────────────────────────────────┐
│  HOST (e.g. Claude Desktop, VS Code, your Python application)   │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │  CLIENT  (one per server connection)                     │   │
│  │  - Establishes and manages the connection                │   │
│  │  - Calls tools/list to discover available tools          │   │
│  │  - Calls tools/call to invoke a specific tool            │   │
│  └──────────────────────────────────────────────────────────┘   │
│           │  JSON-RPC 2.0 over stdio or SSE                     │
│  ┌────────▼─────────────────────────────────────────────────┐   │
│  │  SERVER  (e.g. your finance tools, a database, GitHub)   │   │
│  │  - Exposes tools via tools/list                          │   │
│  │  - Executes tools via tools/call                         │   │
│  │  - Optionally exposes resources and prompts              │   │
│  └──────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

**Host** — the application that owns the user interaction and houses one or more MCP clients. Examples: Claude Desktop, VS Code with GitHub Copilot, your FastAPI agent server.

**Client** — a component inside the host that manages exactly one connection to exactly one MCP server. The client speaks JSON-RPC 2.0. It has no business logic — it is a transport layer.

**Server** — a process that exposes tools (and optionally resources and prompts) over the MCP protocol. It does not know anything about the agent that calls it. It receives a tool name and arguments, executes the tool, and returns the result.

### Transport Layers

MCP supports two transports:

**stdio** — the client launches the server as a subprocess and communicates via stdin/stdout. Used for local tools (filesystem access, local databases, CLI wrappers). Simple to implement; server lifecycle is managed by the client.

**SSE (Server-Sent Events over HTTP)** — the server runs as an HTTP service. The client connects via HTTP POST for requests and SSE for streaming responses. Used for remote tools, shared services, and production deployments behind Azure API Management.

### The Three Primitives

MCP servers expose three types of content:

**Tools** — callable functions with a name, description, and JSON Schema input definition. This is what agents call. Equivalent to LangChain `@tool` and OpenAI function calling.

**Resources** — read-only data sources (files, database records, API responses) that the agent can read without calling a function. Equivalent to RAG retrieval results or file attachments.

**Prompts** — reusable prompt templates with parameterised inputs. The server defines the prompt structure; the client fills in the parameters. Used for standardising how agents approach specific tasks.

### MCP vs LangChain `@tool` — Structural Equivalence

A LangChain `@tool` and an MCP tool definition are structurally identical:

```
LangChain @tool                    MCP Tool Definition
─────────────────────────────────  ─────────────────────────────────
function name          ←→          name
docstring              ←→          description
type-annotated params  ←→          inputSchema (JSON Schema)
return value           ←→          content[0].text in CallToolResult
```

This means any LangChain tool can be exposed as an MCP tool with a thin adapter — and any MCP tool can be imported into LangChain as a `@tool`. This is the bridge this guide builds.

In [4]:
# ── MCP Server — Finance + Healthcare domain tools ────────────────────────────
# This file is written to disk and run as a subprocess (stdio transport).
# It is a standalone Python script — no LangChain dependency.

MCP_SERVER_CODE = '''
#!/usr/bin/env python3
"""
enterprise_mcp_server.py
MCP server exposing finance and healthcare tools over stdio transport.
Run directly: python enterprise_mcp_server.py
Connect from a client: use StdioServerParameters pointing at this file.
"""
import asyncio
import json
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types

app = Server("enterprise-tools")

# ── Tool data (in production: real API calls) ─────────────────────────────────
PRICES = {"AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
          "GS": 452.10, "JNJ": 162.88, "PFE": 29.54}
RATES  = {"fed_funds": "5.25%", "sofr": "5.30%",
          "us_10yr_treasury": "4.48%", "us_2yr_treasury": "4.85%"}
INTERACTIONS = {
    frozenset(["warfarin", "aspirin"]):          ("MAJOR",    "Significantly increases bleeding risk."),
    frozenset(["metformin", "ibuprofen"]):         ("MODERATE", "NSAIDs reduce renal clearance of metformin."),
    frozenset(["atorvastatin", "clarithromycin"]): ("MAJOR",    "CYP3A4 inhibition elevates atorvastatin."),
}
GUIDELINES = {
    "type2_diabetes":      "ADA 2024: First-line metformin + lifestyle (HbA1c < 7%). Second-line GLP-1 or SGLT-2.",
    "hypertension":        "ACC/AHA 2017: Target < 130/80. First-line ACE/ARB, thiazide, or CCB.",
    "atrial_fibrillation": "ESC 2020: CHA2DS2-VASc >= 2 (male) anticoagulate. Prefer NOACs.",
}


# ── tools/list — advertises all available tools to the client ─────────────────
@app.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="get_stock_price",
            description="Retrieve the current stock price for a ticker symbol (AAPL, MSFT, JPM, GS, JNJ, PFE).",
            inputSchema={
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Exchange ticker symbol e.g. AAPL"}
                },
                "required": ["ticker"]
            }
        ),
        types.Tool(
            name="get_interest_rate",
            description="Look up a benchmark interest rate. Supported: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury.",
            inputSchema={
                "type": "object",
                "properties": {
                    "rate_type": {"type": "string", "description": "Rate identifier e.g. fed_funds"}
                },
                "required": ["rate_type"]
            }
        ),
        types.Tool(
            name="calculate_portfolio_value",
            description="Calculate total market value from a comma-separated TICKER:SHARES string e.g. AAPL:100,JPM:50.",
            inputSchema={
                "type": "object",
                "properties": {
                    "holdings": {"type": "string", "description": "Comma-separated TICKER:SHARES pairs"}
                },
                "required": ["holdings"]
            }
        ),
        types.Tool(
            name="check_drug_interaction",
            description="Check for a known clinical interaction between two medications (generic names).",
            inputSchema={
                "type": "object",
                "properties": {
                    "drug_a": {"type": "string", "description": "Generic name of first medication"},
                    "drug_b": {"type": "string", "description": "Generic name of second medication"}
                },
                "required": ["drug_a", "drug_b"]
            }
        ),
        types.Tool(
            name="get_clinical_guideline",
            description="Return clinical guideline summary. Supported conditions: type2_diabetes, hypertension, atrial_fibrillation.",
            inputSchema={
                "type": "object",
                "properties": {
                    "condition": {"type": "string", "description": "Medical condition name"}
                },
                "required": ["condition"]
            }
        ),
    ]


# ── tools/call — executes a tool by name ──────────────────────────────────────
@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    result = _dispatch(name, arguments)
    return [types.TextContent(type="text", text=result)]


def _dispatch(name: str, args: dict) -> str:
    """Route tool calls to the correct implementation."""
    if name == "get_stock_price":
        t = args["ticker"].upper().strip()
        return f"${PRICES[t]:.2f} USD" if t in PRICES else f"Ticker {t} not found."

    elif name == "get_interest_rate":
        r = RATES.get(args["rate_type"].lower().strip())
        return r if r else f"Rate not recognised: {args['rate_type']}"

    elif name == "calculate_portfolio_value":
        total, lines = 0.0, []
        for pair in args["holdings"].split(","):
            t, s = pair.strip().split(":")
            v = PRICES.get(t.upper(), 0.0) * float(s)
            total += v
            lines.append(f"  {t.upper()}: {s} x ${PRICES.get(t.upper(), 0):.2f} = ${v:,.2f}")
        return "\n".join(lines) + f"\nTotal: ${total:,.2f} USD"

    elif name == "check_drug_interaction":
        key = frozenset([args["drug_a"].lower().strip(), args["drug_b"].lower().strip()])
        if key in INTERACTIONS:
            sev, note = INTERACTIONS[key]
            return f"Severity: {sev}\nNote: {note}"
        return f"No major interaction documented between {args[\x27drug_a\x27]} and {args[\x27drug_b\x27]}."

    elif name == "get_clinical_guideline":
        g = GUIDELINES.get(args["condition"].lower().strip())
        return g if g else f"No guideline found for: {args[\x27condition\x27]}"

    else:
        return f"Unknown tool: {name}"


if __name__ == "__main__":
    asyncio.run(stdio_server(app))
'''

# Fix escaped quotes in the dispatch function
MCP_SERVER_CODE = MCP_SERVER_CODE.replace("\\x27", "'")

server_path = Path('enterprise_mcp_server.py')
server_path.write_text(MCP_SERVER_CODE.strip(), encoding='utf-8')
print(f'MCP server written to: {server_path.resolve()}')
print(f'Lines: {len(MCP_SERVER_CODE.splitlines())}')
print()
print('Server exposes 5 tools:')
print('  Finance:    get_stock_price, get_interest_rate, calculate_portfolio_value')
print('  Healthcare: check_drug_interaction, get_clinical_guideline')

MCP server written to: /content/enterprise_mcp_server.py
Lines: 142

Server exposes 5 tools:
  Finance:    get_stock_price, get_interest_rate, calculate_portfolio_value
  Healthcare: check_drug_interaction, get_clinical_guideline


## Section 2 — MCP Server Anatomy

The server written in Cell 4 has four components. Understanding each is essential before building your own.

### 1. Server Initialisation

```python
from mcp.server import Server
app = Server("enterprise-tools")
```

The string `"enterprise-tools"` is the server's identity. Clients use this name to identify the server in multi-server configurations. Use a stable, descriptive name — changing it breaks client configurations.

### 2. `@app.list_tools()` — The Tool Registry

```python
@app.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="get_stock_price",
            description="Retrieve the current stock price...",
            inputSchema={"type": "object", "properties": {...}, "required": [...]}
        )
    ]
```

This is the **discovery endpoint**. When a client connects, the first thing it does is call `tools/list`. The server returns every tool it exposes. The client (and the LLM behind it) uses the `description` to decide when to call each tool. **Write descriptions as if writing API documentation for a developer who has never seen your code.**

The `inputSchema` is standard JSON Schema. The LLM uses it to construct valid arguments. Always mark required fields in the `required` array.

### 3. `@app.call_tool()` — The Execution Endpoint

```python
@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    result = _dispatch(name, arguments)
    return [types.TextContent(type="text", text=result)]
```

This receives the tool name and arguments, executes the tool, and returns the result wrapped in a `TextContent` object. The client receives this and injects the `text` value into the agent's context as a tool result.

Always return `list[types.TextContent]` — the return type is a list because MCP supports multi-part responses (e.g. text + image).

### 4. Transport Entry Point

```python
if __name__ == "__main__":
    asyncio.run(stdio_server(app))
```

`stdio_server` attaches the MCP protocol to stdin/stdout. The client launches this script as a subprocess and reads/writes JSON-RPC messages through the standard streams. For HTTP/SSE transport, replace `stdio_server` with an SSE server from the `mcp.server.sse` module.

### Converting a LangChain `@tool` to MCP

The mapping is mechanical:

```python
# LangChain @tool
@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve the current stock price for a ticker (e.g. AAPL)."""
    ...

# Equivalent MCP Tool definition
types.Tool(
    name="get_stock_price",
    description="Retrieve the current stock price for a ticker (e.g. AAPL).",
    inputSchema={
        "type": "object",
        "properties": {
            "ticker": {"type": "string", "description": "Exchange ticker symbol"}
        },
        "required": ["ticker"]
    }
)
```

The function name becomes `name`. The docstring becomes `description`. Each typed parameter becomes a JSON Schema property. That is the complete conversion.

In [7]:
%pip install nest_asyncio fastapi uvicorn httpx --upgrade --quiet

import nest_asyncio
nest_asyncio.apply()

import asyncio, threading, time, socket, os, uuid
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import httpx
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)

llm = AzureChatOpenAI(
    azure_deployment=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    temperature=0, max_tokens=4096,
)

# ── Tool implementations ──────────────────────────────────────────────────────
PRICES = {'AAPL': 189.45, 'MSFT': 415.20, 'JPM': 198.73,
          'GS': 452.10, 'JNJ': 162.88, 'PFE': 29.54}
RATES  = {'fed_funds': '5.25%', 'sofr': '5.30%',
          'us_10yr_treasury': '4.48%', 'us_2yr_treasury': '4.85%'}
INTERACTIONS = {
    frozenset(['warfarin',     'aspirin']):          ('MAJOR',    'Significantly increases bleeding risk.'),
    frozenset(['metformin',    'ibuprofen']):         ('MODERATE', 'NSAIDs reduce renal clearance of metformin.'),
    frozenset(['atorvastatin', 'clarithromycin']):    ('MAJOR',    'CYP3A4 inhibition elevates atorvastatin.'),
}
GUIDELINES = {
    'type2_diabetes':      'ADA 2024: First-line metformin (HbA1c < 7%). Second-line GLP-1 or SGLT-2.',
    'hypertension':        'ACC/AHA 2017: Target < 130/80. First-line ACE/ARB, thiazide, or CCB.',
    'atrial_fibrillation': 'ESC 2020: CHA2DS2-VASc >= 2 (male) anticoagulate. Prefer NOACs.',
}

def _get_stock_price(ticker: str) -> str:
    t = ticker.upper().strip()
    return f'${PRICES[t]:.2f} USD' if t in PRICES else f'Ticker {t} not found.'

def _get_interest_rate(rate_type: str) -> str:
    r = RATES.get(rate_type.lower().strip())
    return r if r else f"Rate '{rate_type}' not recognised."

def _calculate_portfolio_value(holdings: str) -> str:
    total, lines = 0.0, []
    for pair in holdings.split(','):
        t, s = pair.strip().split(':')
        v = PRICES.get(t.upper(), 0.0) * float(s)
        total += v
        lines.append(f'  {t.upper()}: {s} shares = ${v:,.2f}')
    return '\n'.join(lines) + f'\nTotal: ${total:,.2f} USD'

def _check_drug_interaction(drug_a: str, drug_b: str) -> str:
    key = frozenset([drug_a.lower().strip(), drug_b.lower().strip()])
    if key in INTERACTIONS:
        sev, note = INTERACTIONS[key]
        return f'Severity: {sev}\nNote: {note}'
    return f'No major interaction documented between {drug_a} and {drug_b}.'

def _get_clinical_guideline(condition: str) -> str:
    g = GUIDELINES.get(condition.lower().strip())
    return g if g else f"No guideline found for '{condition}'."

# ── LangChain agents (for the /agent endpoint) ────────────────────────────────
@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve current stock price for a ticker symbol e.g. AAPL, MSFT, JPM."""
    return _get_stock_price(ticker)

@tool
def get_interest_rate(rate_type: str) -> str:
    """Look up a benchmark interest rate: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury."""
    return _get_interest_rate(rate_type)

@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """Check for a known clinical interaction between two medications (generic names)."""
    return _check_drug_interaction(drug_a, drug_b)

@tool
def get_clinical_guideline(condition: str) -> str:
    """Return clinical guideline. Supported: type2_diabetes, hypertension, atrial_fibrillation."""
    return _get_clinical_guideline(condition)

finance_agent = create_react_agent(
    model=llm,
    tools=[get_stock_price, get_interest_rate],
    prompt='You are a senior financial analyst. Use tools for data. Never guess.',
    checkpointer=MemorySaver(),
)
clinical_agent = create_react_agent(
    model=llm,
    tools=[check_drug_interaction, get_clinical_guideline],
    prompt='You are a clinical decision-support assistant. Always include a disclaimer.',
    checkpointer=MemorySaver(),
)

TOOL_REGISTRY = {
    'get_stock_price':         lambda a: _get_stock_price(a['ticker']),
    'get_interest_rate':       lambda a: _get_interest_rate(a['rate_type']),
    'calculate_portfolio_value': lambda a: _calculate_portfolio_value(a['holdings']),
    'check_drug_interaction':  lambda a: _check_drug_interaction(a['drug_a'], a['drug_b']),
    'get_clinical_guideline':  lambda a: _get_clinical_guideline(a['condition']),
}

MCP_TOOLS_SCHEMA = [
    {'name': 'get_stock_price',
     'description': 'Retrieve current stock price for a ticker symbol (AAPL, MSFT, JPM, GS, JNJ, PFE).',
     'inputSchema': {'type': 'object', 'properties': {'ticker': {'type': 'string'}}, 'required': ['ticker']}},
    {'name': 'get_interest_rate',
     'description': 'Look up a benchmark rate: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury.',
     'inputSchema': {'type': 'object', 'properties': {'rate_type': {'type': 'string'}}, 'required': ['rate_type']}},
    {'name': 'calculate_portfolio_value',
     'description': "Calculate portfolio value from TICKER:SHARES pairs e.g. 'AAPL:100,JPM:50'.",
     'inputSchema': {'type': 'object', 'properties': {'holdings': {'type': 'string'}}, 'required': ['holdings']}},
    {'name': 'check_drug_interaction',
     'description': 'Check for a known clinical interaction between two medications.',
     'inputSchema': {'type': 'object', 'properties': {
         'drug_a': {'type': 'string'}, 'drug_b': {'type': 'string'}},
         'required': ['drug_a', 'drug_b']}},
    {'name': 'get_clinical_guideline',
     'description': 'Return clinical guideline. Supported: type2_diabetes, hypertension, atrial_fibrillation.',
     'inputSchema': {'type': 'object', 'properties': {'condition': {'type': 'string'}}, 'required': ['condition']}},
]


# ── FastAPI app — MCP-compatible REST API ─────────────────────────────────────
class ToolCallRequest(BaseModel):
    tool:      str
    arguments: dict

class AgentRequest(BaseModel):
    question: str
    domain:   str = 'finance'   # 'finance' or 'clinical'
    thread_id: str = None

mcp_app = FastAPI(title='MCP Tool Server', version='1.0.0',
                  description='MCP-compatible REST API for enterprise finance and healthcare tools.')

@mcp_app.get('/health')
def health():
    return {'status': 'ok', 'tools': len(MCP_TOOLS_SCHEMA)}

@mcp_app.get('/tools/list')
def tools_list():
    """MCP tools/list equivalent — returns all available tools and their schemas."""
    return {'tools': MCP_TOOLS_SCHEMA}

@mcp_app.post('/tools/call')
def tools_call(req: ToolCallRequest):
    """MCP tools/call equivalent — executes a tool by name with provided arguments."""
    fn = TOOL_REGISTRY.get(req.tool)
    if not fn:
        return JSONResponse(status_code=404,
                            content={'error': f"Tool '{req.tool}' not found.",
                                     'available': list(TOOL_REGISTRY.keys())})
    try:
        result = fn(req.arguments)
        return {'tool': req.tool, 'content': [{'type': 'text', 'text': result}], 'status': 'ok'}
    except KeyError as e:
        return JSONResponse(status_code=422,
                            content={'error': f'Missing argument: {e}', 'tool': req.tool})
    except Exception as e:
        return JSONResponse(status_code=500,
                            content={'error': str(e), 'tool': req.tool})

@mcp_app.post('/agent/invoke')
def agent_invoke(req: AgentRequest):
    """Invoke a full LangChain ReAct agent and return its final answer."""
    agent = finance_agent if req.domain == 'finance' else clinical_agent
    tid   = req.thread_id or str(uuid.uuid4())
    cfg   = {'configurable': {'thread_id': tid}, 'recursion_limit': 10}
    try:
        result = agent.invoke({'messages': [HumanMessage(content=req.question)]}, config=cfg)
        answer = result['messages'][-1].content
        return {'answer': answer, 'thread_id': tid, 'domain': req.domain,
                'steps': len(result['messages']), 'status': 'ok'}
    except Exception as e:
        return JSONResponse(status_code=500, content={'error': str(e), 'domain': req.domain})


# ── Start server ──────────────────────────────────────────────────────────────
import uvicorn

def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(mcp_app, host='127.0.0.1', port=8200, log_level='warning')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

threading.Thread(target=_run_server, daemon=True).start()

for _ in range(20):
    try:
        with socket.create_connection(('127.0.0.1', 8200), timeout=0.5): break
    except OSError:
        time.sleep(0.5)

BASE = 'http://127.0.0.1:8200'
print(f'Server ready at {BASE}')
print(f'  GET  /health       — liveness check')
print(f'  GET  /tools/list   — discover all tools')
print(f'  POST /tools/call   — call any tool')
print(f'  POST /agent/invoke — invoke a full ReAct agent')
print(f'  GET  /docs         — OpenAPI UI')

# ── Tests ─────────────────────────────────────────────────────────────────────
print('\n=== Health ===')
print(httpx.get(f'{BASE}/health').json())

print('\n=== Tool Discovery (/tools/list) ===')
tools = httpx.get(f'{BASE}/tools/list').json()['tools']
for t in tools:
    print(f"  {t['name']:<35} {t['description'][:55]}...")

print('\n=== Tool Calls (/tools/call) ===')
calls = [
    ('get_stock_price',          {'ticker': 'AAPL'}),
    ('get_interest_rate',        {'rate_type': 'sofr'}),
    ('calculate_portfolio_value',{'holdings': 'AAPL:100,JPM:50'}),
    ('check_drug_interaction',   {'drug_a': 'warfarin', 'drug_b': 'aspirin'}),
    ('get_clinical_guideline',   {'condition': 'type2_diabetes'}),
]
for tool_name, args in calls:
    r = httpx.post(f'{BASE}/tools/call', json={'tool': tool_name, 'arguments': args})
    text = r.json()['content'][0]['text']
    print(f'  {tool_name}:')
    for line in text.split('\n'):
        print(f'    {line}')

print('\n=== Agent Invoke (/agent/invoke) ===')
r = httpx.post(f'{BASE}/agent/invoke',
               json={'question': 'What is the current price of AAPL and the Fed Funds rate?',
                     'domain': 'finance'},
               timeout=60.0)
data = r.json()
print(f"  Domain: {data['domain']}  Steps: {data['steps']}  Thread: {data['thread_id'][:16]}...")
print(f"  Answer: {data['answer'][:300]}...")

<frozen posixpath>:82: RuntimeWarning: coroutine 'demo_client' was never awaited
/tmp/ipykernel_255/2739565618.py:93: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  finance_agent = create_react_agent(
/tmp/ipykernel_255/2739565618.py:99: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  clinical_agent = create_react_agent(


Server ready at http://127.0.0.1:8200
  GET  /health       — liveness check
  GET  /tools/list   — discover all tools
  POST /tools/call   — call any tool
  POST /agent/invoke — invoke a full ReAct agent
  GET  /docs         — OpenAPI UI

=== Health ===
{'status': 'ok', 'tools': 5}

=== Tool Discovery (/tools/list) ===
  get_stock_price                     Retrieve current stock price for a ticker symbol (AAPL,...
  get_interest_rate                   Look up a benchmark rate: fed_funds, sofr, us_10yr_trea...
  calculate_portfolio_value           Calculate portfolio value from TICKER:SHARES pairs e.g....
  check_drug_interaction              Check for a known clinical interaction between two medi...
  get_clinical_guideline              Return clinical guideline. Supported: type2_diabetes, h...

=== Tool Calls (/tools/call) ===
  get_stock_price:
    $189.45 USD
  get_interest_rate:
    5.30%
  calculate_portfolio_value:
      AAPL: 100 shares = $18,945.00
      JPM: 50 shares = $9,936

## Section 3 — Converting LangChain Agents to MCP Servers

A LangChain agent can itself be exposed as an MCP tool. This is the most powerful pattern for enterprise systems: wrap an entire multi-step reasoning agent as a single MCP tool, and any MCP-compatible client (Claude Desktop, Copilot Agent Mode, another LangGraph agent) can invoke it without knowing anything about LangChain.

### Pattern: Agent-as-Tool

```python
# The agent is defined normally in LangChain
finance_agent = create_react_agent(model=llm, tools=finance_tools, ...)

# It is exposed as a single MCP tool
types.Tool(
    name="finance_analyst",
    description="""
        A senior financial analyst agent with access to live stock prices,
        interest rates, and portfolio calculation tools.
        Input: a natural-language financial question.
        Returns: a detailed analysis with cited data.
    """,
    inputSchema={
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The financial question to analyse"}
        },
        "required": ["question"]
    }
)
```

When `call_tool` receives `name="finance_analyst"`, it invokes the LangChain agent, waits for the full response, and returns the final answer as `TextContent`.

### Why This Matters

Before MCP: your LangChain agent was only callable from Python code that imported LangChain.

After MCP: your LangChain agent is callable from Claude Desktop, GitHub Copilot Agent Mode, any VS Code extension, any application that speaks MCP — without those applications knowing anything about LangChain, Python, or your internal architecture.

The agent is now a **service** with a stable, documented interface.

In [8]:
# ── MCP server that wraps LangChain agents as MCP tools ───────────────────────
# This is the agent-as-tool pattern: each agent is exposed as one MCP tool.
# A caller (Claude Desktop, Copilot, another agent) calls the tool with a
# natural-language question and gets back the agent's full reasoned response.

AGENT_SERVER_CODE = '''
#!/usr/bin/env python3
"""
agent_mcp_server.py
MCP server that exposes LangChain ReAct agents as MCP tools.
Callers send a natural-language question; the agent runs its full
ReAct loop and returns the final answer.
"""
import asyncio, os, uuid
from dotenv import load_dotenv
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# ── Build the agents once at server startup ───────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0, max_tokens=4096,
)

@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve current stock price for a ticker symbol."""
    prices = {"AAPL": "189.45", "MSFT": "415.20", "JPM": "198.73",
              "GS": "452.10", "JNJ": "162.88", "PFE": "29.54"}
    t = ticker.upper().strip()
    return f"${prices[t]} USD" if t in prices else f"Ticker {t} not found."

@tool
def get_interest_rate(rate_type: str) -> str:
    """Look up a benchmark interest rate."""
    rates = {"fed_funds": "5.25%", "sofr": "5.30%",
             "us_10yr_treasury": "4.48%", "us_2yr_treasury": "4.85%"}
    return rates.get(rate_type.lower(), f"Rate {rate_type} not recognised.")

@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """Check for known interaction between two medications."""
    db = {
        frozenset(["warfarin", "aspirin"]): ("MAJOR", "Increases bleeding risk."),
        frozenset(["metformin", "ibuprofen"]): ("MODERATE", "NSAIDs reduce renal clearance."),
    }
    key = frozenset([drug_a.lower(), drug_b.lower()])
    sev, note = db.get(key, (None, None))
    return f"Severity: {sev}. {note}" if sev else f"No major interaction documented."

@tool
def get_clinical_guideline(condition: str) -> str:
    """Return clinical guideline for a condition."""
    gl = {
        "type2_diabetes": "ADA 2024: First-line metformin (HbA1c < 7%). Second-line GLP-1 or SGLT-2.",
        "hypertension": "ACC/AHA 2017: Target < 130/80. First-line ACE/ARB, thiazide, or CCB.",
    }
    return gl.get(condition.lower(), f"No guideline found for {condition}.")

# Each agent is built once and reused across all calls
FINANCE_AGENT = create_react_agent(
    model=llm,
    tools=[get_stock_price, get_interest_rate],
    prompt="You are a senior financial analyst. Use tools for data. Never guess.",
    checkpointer=MemorySaver(),
)
CLINICAL_AGENT = create_react_agent(
    model=llm,
    tools=[check_drug_interaction, get_clinical_guideline],
    prompt="You are a clinical decision-support assistant. Always include a disclaimer.",
    checkpointer=MemorySaver(),
)


def _run_agent(agent, question: str) -> str:
    """Invoke a LangChain agent synchronously and return its final answer."""
    cfg    = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10}
    result = agent.invoke({"messages": [HumanMessage(content=question)]}, config=cfg)
    return result["messages"][-1].content


app = Server("agent-tools")

@app.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="finance_analyst",
            description=(
                "A senior financial analyst agent with live stock price and interest rate tools. "
                "Input a natural-language financial question. "
                "Returns a detailed analysis with cited data. "
                "Use for: stock prices, rate lookups, portfolio questions, macro analysis."
            ),
            inputSchema={
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The financial question to analyse"}
                },
                "required": ["question"]
            }
        ),
        types.Tool(
            name="clinical_analyst",
            description=(
                "A clinical decision-support agent with drug interaction and clinical guideline tools. "
                "Input a natural-language clinical question. "
                "Returns evidence-based clinical information with a professional disclaimer. "
                "Use for: drug interactions, treatment guidelines, clinical protocols."
            ),
            inputSchema={
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The clinical question to answer"}
                },
                "required": ["question"]
            }
        ),
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    if name == "finance_analyst":
        answer = _run_agent(FINANCE_AGENT, arguments["question"])
    elif name == "clinical_analyst":
        answer = _run_agent(CLINICAL_AGENT, arguments["question"])
    else:
        answer = f"Unknown tool: {name}"
    return [types.TextContent(type="text", text=answer)]


if __name__ == "__main__":
    asyncio.run(stdio_server(app))
'''

Path('agent_mcp_server.py').write_text(AGENT_SERVER_CODE.strip(), encoding='utf-8')
print('agent_mcp_server.py written.')
print()
print('This server exposes 2 MCP tools:')
print('  finance_analyst  — wraps the full LangChain finance ReAct agent')
print('  clinical_analyst — wraps the full LangChain clinical ReAct agent')
print()
print('Any MCP client calling finance_analyst({"question": "..."}) triggers')
print('a full multi-step LangChain ReAct loop internally.')

agent_mcp_server.py written.

This server exposes 2 MCP tools:
  finance_analyst  — wraps the full LangChain finance ReAct agent
  clinical_analyst — wraps the full LangChain clinical ReAct agent

Any MCP client calling finance_analyst({"question": "..."}) triggers
a full multi-step LangChain ReAct loop internally.


In [11]:
from langchain_core.tools import StructuredTool
import httpx

BASE = 'http://127.0.0.1:8200'

def _call_tool(base_url: str, name: str, **kwargs) -> str:
    r = httpx.post(f'{base_url}/tools/call',
                   json={'tool': name, 'arguments': kwargs},
                   timeout=30.0)
    data = r.json()
    # Show full response if something is wrong
    if 'content' not in data:
        return f"Server error: {data}"
    return data['content'][0]['text']


def load_tools_from_http_server(base_url: str) -> list:
    mcp_tools = httpx.get(f'{base_url}/tools/list').json()['tools']
    lc_tools  = []
    for t in mcp_tools:
        tool_name = t['name']
        def _invoke(base_url=base_url, name=tool_name, **kwargs) -> str:
            return _call_tool(base_url, name, **kwargs)
        lc_tools.append(StructuredTool.from_function(
            func=_invoke,
            name=tool_name,
            description=t['description'],
        ))
        print(f'  Loaded: {tool_name}')
    return lc_tools


print('=== Loading tools ===')
mcp_lc_tools = load_tools_from_http_server(BASE)
tool_map = {t.name: t for t in mcp_lc_tools}

# Debug: test the raw HTTP call first
print('\n=== Raw HTTP test ===')
raw = httpx.post(f'{BASE}/tools/call',
                 json={'tool': 'get_stock_price', 'arguments': {'ticker': 'MSFT'}})
print(f'Status: {raw.status_code}')
print(f'Body:   {raw.json()}')

print('\n=== Tool invocations ===')
r1 = tool_map['get_stock_price'].invoke({'ticker': 'MSFT'})
print(f'  get_stock_price(MSFT):    {r1}')

r2 = tool_map['get_interest_rate'].invoke({'rate_type': 'sofr'})
print(f'  get_interest_rate(sofr):  {r2}')

r3 = tool_map['calculate_portfolio_value'].invoke({'holdings': 'AAPL:100,JPM:50'})
print(f'  portfolio value:\n    {r3}')

r4 = tool_map['check_drug_interaction'].invoke({'drug_a': 'metformin', 'drug_b': 'ibuprofen'})
print(f'  drug interaction:\n    {r4}')

r5 = tool_map['get_clinical_guideline'].invoke({'condition': 'atrial_fibrillation'})
print(f'  guideline:  {r5}')

print('\n=== Agent using MCP-loaded tools ===')
mcp_agent = create_react_agent(
    model=llm,
    tools=mcp_lc_tools,
    prompt='You are a senior analyst. Use tools for all data. Never guess.',
    checkpointer=MemorySaver(),
)

cfg    = {'configurable': {'thread_id': str(uuid.uuid4())}, 'recursion_limit': 10}
result = mcp_agent.invoke(
    {'messages': [HumanMessage(
        content='What is the current price of JNJ and is there an interaction '
                'between warfarin and aspirin?'
    )]},
    config=cfg,
)
print(result['messages'][-1].content)

=== Loading tools ===
  Loaded: get_stock_price
  Loaded: get_interest_rate
  Loaded: calculate_portfolio_value
  Loaded: check_drug_interaction
  Loaded: get_clinical_guideline

=== Raw HTTP test ===
Status: 200
Body:   {'tool': 'get_stock_price', 'content': [{'type': 'text', 'text': '$415.20 USD'}], 'status': 'ok'}

=== Tool invocations ===
  get_stock_price(MSFT):    Server error: {'error': "Missing argument: 'ticker'", 'tool': 'get_stock_price'}
  get_interest_rate(sofr):  Server error: {'error': "Missing argument: 'rate_type'", 'tool': 'get_interest_rate'}
  portfolio value:
    Server error: {'error': "Missing argument: 'holdings'", 'tool': 'calculate_portfolio_value'}
  drug interaction:
    Server error: {'error': "Missing argument: 'drug_a'", 'tool': 'check_drug_interaction'}
  guideline:  Server error: {'error': "Missing argument: 'condition'", 'tool': 'get_clinical_guideline'}

=== Agent using MCP-loaded tools ===


/tmp/ipykernel_255/2209138408.py:61: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mcp_agent = create_react_agent(


I am unable to retrieve the current price of JNJ or check the interaction between warfarin and aspirin due to repeated technical issues. I recommend trying again later.


## Section 4 — Multi-Server Composition

A single MCP client can connect to multiple servers simultaneously and merge their tool lists. This is the enterprise pattern: each team owns and deploys their own MCP server; a central supervisor agent connects to all of them.

```
Supervisor Agent
    ├── Client A ──→ finance_mcp_server    (get_stock_price, get_rate, ...)
    ├── Client B ──→ clinical_mcp_server   (check_drug_interaction, ...)
    ├── Client C ──→ rag_mcp_server        (retrieve_policy, ...)
    └── Client D ──→ crm_mcp_server        (get_customer, update_case, ...)
```

The supervisor sees all tools from all servers as a flat list. It does not know or care which server each tool comes from. Adding a new capability means deploying a new MCP server — the supervisor requires zero code changes.

### Key Design Decisions for Multi-Server Systems

**Tool name uniqueness:** Tool names must be unique across all connected servers. If two servers expose a tool named `search`, the client cannot disambiguate. Use namespacing: `finance_search`, `clinical_search`.

**Server lifecycle:** In stdio transport, each server is a subprocess. The client manages startup and shutdown. If a server crashes, the client should detect the broken pipe and either restart the server or remove its tools from the available set.

**Authentication per server:** Each server can require its own authentication. The client passes auth credentials in the `initialize` call or in per-request headers (SSE transport). In Azure, use Managed Identity per server — the client passes the identity token, and each server validates it independently.

**Timeout handling:** Each tool call has a network round trip plus execution time. Set per-tool timeouts. If a server becomes slow, its tools should degrade gracefully (return a timeout error as `TextContent`) rather than blocking the agent indefinitely.

### MCP in Production: Azure Deployment

For production SSE-transport MCP servers on Azure:

1. **Container:** Package the MCP server as a Docker container. Deploy to Azure Container Apps.
2. **Auth:** Add Azure AD authentication as middleware. The client passes an AAD token in the `Authorization` header.
3. **API Management:** Put Azure API Management in front of all MCP servers. APIM adds rate limiting, logging, and a developer portal for tool documentation.
4. **Monitoring:** The MCP server emits structured logs per tool call. Forward to Azure Monitor Log Analytics. Alert on error rate > 1% or p95 latency > 5s.

In [13]:
import httpx

# ── Two HTTP MCP servers ───────────────────────────────────────────────────────
# Both run on the same FastAPI app (port 8200) in this lab.
# In production each would be a separate service at its own URL.
SERVER_CONFIGS = [
    {'label': 'finance-tools',   'base_url': BASE, 'tool_filter': ['get_stock_price', 'get_interest_rate', 'calculate_portfolio_value']},
    {'label': 'clinical-tools',  'base_url': BASE, 'tool_filter': ['check_drug_interaction', 'get_clinical_guideline']},
]

def discover_server_tools(base_url: str, tool_filter: list = None) -> list[dict]:
    """Fetch tool list from one HTTP MCP server, optionally filtering by name."""
    resp  = httpx.get(f'{base_url}/tools/list', timeout=10.0)
    tools = resp.json()['tools']
    if tool_filter:
        tools = [t for t in tools if t['name'] in tool_filter]
    return tools

def call_server_tool(base_url: str, tool_name: str, arguments: dict) -> str:
    """Call one tool on one HTTP MCP server."""
    resp = httpx.post(f'{base_url}/tools/call',
                      json={'tool': tool_name, 'arguments': arguments},
                      timeout=30.0)
    data = resp.json()
    if 'content' not in data:
        return f"Error: {data.get('error', data)}"
    return data['content'][0]['text']

def call_agent(base_url: str, domain: str, question: str) -> str:
    """Call the /agent/invoke endpoint on an HTTP MCP server."""
    resp = httpx.post(f'{base_url}/agent/invoke',
                      json={'question': question, 'domain': domain},
                      timeout=60.0)
    data = resp.json()
    if 'answer' not in data:
        return f"Error: {data.get('error', data)}"
    return data['answer']


# ── Step 1: Discover tools from all servers ────────────────────────────────────
print('=== Multi-Server Tool Discovery ===')
tool_registry = {}   # name → {tool_schema, base_url, server_label}

for cfg in SERVER_CONFIGS:
    label    = cfg['label']
    base_url = cfg['base_url']
    try:
        tools = discover_server_tools(base_url, cfg.get('tool_filter'))
        for t in tools:
            tool_registry[t['name']] = {
                'name':         t['name'],
                'description':  t['description'],
                'inputSchema':  t['inputSchema'],
                'base_url':     base_url,
                'server_label': label,
            }
        print(f'  [{label}] {len(tools)} tools: {[t["name"] for t in tools]}')
    except Exception as e:
        print(f'  [{label}] FAILED: {e}')

print(f'\nTotal tools in registry: {len(tool_registry)}')


# ── Step 2: Cross-server tool calls ──────────────────────────────────────────
print('\n=== Cross-Server Tool Calls ===')

# Finance tool
t1 = tool_registry['get_stock_price']
r1 = call_server_tool(t1['base_url'], t1['name'], {'ticker': 'JNJ'})
print(f'  [{t1["server_label"]}] get_stock_price(JNJ): {r1}')

# Portfolio tool
t2 = tool_registry['calculate_portfolio_value']
r2 = call_server_tool(t2['base_url'], t2['name'], {'holdings': 'JNJ:100,PFE:200'})
print(f'  [{t2["server_label"]}] calculate_portfolio_value:')
for line in r2.split('\n'):
    print(f'    {line}')

# Clinical tool
t3 = tool_registry['check_drug_interaction']
r3 = call_server_tool(t3['base_url'], t3['name'], {'drug_a': 'warfarin', 'drug_b': 'aspirin'})
print(f'  [{t3["server_label"]}] check_drug_interaction(warfarin, aspirin):')
for line in r3.split('\n'):
    print(f'    {line}')


# ── Step 3: Agent calls across domains ────────────────────────────────────────
print('\n=== Agent Calls via /agent/invoke ===')

print('\n  [finance-tools] finance_analyst:')
r4 = call_agent(BASE, 'finance',
                'What is the current SOFR rate and how does it affect floating rate bonds?')
print(f'    {r4[:400]}...' if len(r4) > 400 else f'    {r4}')

print('\n  [clinical-tools] clinical_analyst:')
r5 = call_agent(BASE, 'clinical',
                'Is there an interaction between atorvastatin and clarithromycin?')
print(f'    {r5[:400]}...' if len(r5) > 400 else f'    {r5}')


# ── Step 4: Build LangChain agent from aggregated registry ────────────────────
print('\n=== LangChain Agent from Aggregated Tool Registry ===')

aggregated_lc_tools = []
for entry in tool_registry.values():
    tool_name = entry['name']
    tool_url  = entry['base_url']

    def _invoke(base_url=tool_url, name=tool_name, **kwargs) -> str:
        return call_server_tool(base_url, name, kwargs)

    aggregated_lc_tools.append(StructuredTool.from_function(
        func=_invoke,
        name=tool_name,
        description=entry['description'],
    ))

multi_server_agent = create_react_agent(
    model=llm,
    tools=aggregated_lc_tools,
    prompt=(
        'You are a senior healthcare investment analyst. '
        'You have access to live market data tools and clinical tools. '
        'Use tools for all facts. Never guess prices, rates, or clinical data.'
    ),
    checkpointer=MemorySaver(),
)

cfg    = {'configurable': {'thread_id': str(uuid.uuid4())}, 'recursion_limit': 12}
result = multi_server_agent.invoke(
    {'messages': [HumanMessage(
        content='Evaluate Pfizer (PFE): give me the current stock price, '
                'the Fed Funds rate context, and whether metformin-ibuprofen '
                'interaction is material to their diabetes pipeline.'
    )]},
    config=cfg,
)
msgs       = result['messages']
tools_used = [tc['name'] for m in msgs for tc in (getattr(m, 'tool_calls', []) or [])]
print(f'Tools used: {tools_used}')
print(f'Steps: {len(msgs)}')
print(result['messages'][-1].content)

=== Multi-Server Tool Discovery ===
  [finance-tools] 3 tools: ['get_stock_price', 'get_interest_rate', 'calculate_portfolio_value']
  [clinical-tools] 2 tools: ['check_drug_interaction', 'get_clinical_guideline']

Total tools in registry: 5

=== Cross-Server Tool Calls ===
  [finance-tools] get_stock_price(JNJ): $162.88 USD
  [finance-tools] calculate_portfolio_value:
      JNJ: 100 shares = $16,288.00
      PFE: 200 shares = $5,908.00
    Total: $22,196.00 USD
  [clinical-tools] check_drug_interaction(warfarin, aspirin):
    Severity: MAJOR
    Note: Significantly increases bleeding risk.

=== Agent Calls via /agent/invoke ===

  [finance-tools] finance_analyst:
    The current Secured Overnight Financing Rate (SOFR) is 5.30%.

### Impact on Floating Rate Bonds:
Floating rate bonds typically have interest payments tied to a benchmark rate like SOFR. When SOFR increases, the interest payments on these bonds also rise, making them more attractive to investors seeking higher yields. Con

/tmp/ipykernel_255/3471214851.py:118: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  multi_server_agent = create_react_agent(


Tools used: ['get_stock_price', 'get_interest_rate', 'check_drug_interaction', 'get_stock_price', 'get_interest_rate', 'check_drug_interaction', 'get_stock_price', 'get_interest_rate', 'check_drug_interaction', 'get_stock_price', 'get_interest_rate', 'check_drug_interaction']
Steps: 18
It seems there is a technical issue with retrieving the data. I recommend revisiting this query later or contacting technical support for assistance.


In [14]:
# ── Production MCP patterns: schema validation, error handling, audit logging ──

import jsonschema
import logging
import time

logging.basicConfig(level=logging.INFO,
                    format='{"time": "%(asctime)s", "level": "%(levelname)s", %(message)s}',
                    datefmt='%Y-%m-%dT%H:%M:%SZ')
mcp_logger = logging.getLogger('mcp.audit')


# ── Pattern 1: Input schema validation before tool execution ───────────────────
TOOL_SCHEMAS = {
    'get_stock_price': {
        'type': 'object',
        'properties': {'ticker': {'type': 'string', 'minLength': 1, 'maxLength': 5}},
        'required': ['ticker']
    },
    'check_drug_interaction': {
        'type': 'object',
        'properties': {
            'drug_a': {'type': 'string', 'minLength': 2},
            'drug_b': {'type': 'string', 'minLength': 2},
        },
        'required': ['drug_a', 'drug_b']
    },
}

def validate_tool_input(tool_name: str, arguments: dict) -> tuple[bool, str]:
    """Validate arguments against the tool's JSON Schema. Returns (valid, error_message)."""
    schema = TOOL_SCHEMAS.get(tool_name)
    if not schema:
        return True, ''   # no schema = no validation
    try:
        jsonschema.validate(instance=arguments, schema=schema)
        return True, ''
    except jsonschema.ValidationError as e:
        return False, e.message


# ── Pattern 2: Retry wrapper for unreliable external APIs ──────────────────────
def with_retry(func, max_attempts: int = 3, delay_s: float = 1.0):
    """Retry a tool call up to max_attempts times on exception."""
    def wrapper(*args, **kwargs):
        last_exc = None
        for attempt in range(1, max_attempts + 1):
            try:
                return func(*args, **kwargs)
            except Exception as exc:
                last_exc = exc
                if attempt < max_attempts:
                    time.sleep(delay_s)
        return f'Tool failed after {max_attempts} attempts: {last_exc}'
    return wrapper


# ── Pattern 3: Audit log every MCP tool call ───────────────────────────────────
def audit_tool_call(tool_name: str, arguments: dict, result: str, latency_ms: float, status: str):
    mcp_logger.info(
        f'"tool": "{tool_name}", '
        f'"arg_keys": {json.dumps(list(arguments.keys()))}, '
        f'"latency_ms": {round(latency_ms, 1)}, '
        f'"status": "{status}", '
        f'"result_len": {len(result)}'
    )


# ── Pattern 4: Safe MCP tool dispatcher combining all three patterns ───────────
def safe_dispatch(tool_name: str, arguments: dict, dispatch_fn) -> str:
    """
    Production-grade tool dispatcher:
    1. Validates input schema
    2. Executes with timing
    3. Emits audit log
    4. Returns structured error on failure
    """
    # Schema validation
    valid, err = validate_tool_input(tool_name, arguments)
    if not valid:
        audit_tool_call(tool_name, arguments, f'INVALID: {err}', 0, 'schema_error')
        return f'Input validation failed: {err}'

    t0 = time.monotonic()
    status = 'ok'
    try:
        result = dispatch_fn(tool_name, arguments)
    except Exception as exc:
        result = f'Tool execution error: {exc}'
        status = 'error'

    latency = (time.monotonic() - t0) * 1000
    audit_tool_call(tool_name, arguments, result, latency, status)
    return result


# ── Demonstration ──────────────────────────────────────────────────────────────
def mock_dispatch(name, args):
    if name == 'get_stock_price':
        prices = {'AAPL': '$189.45', 'MSFT': '$415.20'}
        return prices.get(args['ticker'].upper(), 'Not found')
    if name == 'check_drug_interaction':
        return 'MAJOR interaction: increases bleeding risk.'
    return 'Unknown tool'

print('=== Production Pattern Tests ===')
print()

print('Test 1 — Valid call:')
r = safe_dispatch('get_stock_price', {'ticker': 'AAPL'}, mock_dispatch)
print(f'  Result: {r}')

print('\nTest 2 — Schema violation (ticker too long):')
r = safe_dispatch('get_stock_price', {'ticker': 'TOOLONGTICKER'}, mock_dispatch)
print(f'  Result: {r}')

print('\nTest 3 — Missing required field:')
r = safe_dispatch('check_drug_interaction', {'drug_a': 'warfarin'}, mock_dispatch)
print(f'  Result: {r}')

print('\nTest 4 — Valid drug interaction call:')
r = safe_dispatch('check_drug_interaction', {'drug_a': 'warfarin', 'drug_b': 'aspirin'}, mock_dispatch)
print(f'  Result: {r}')

print()
print('Check above for JSON audit log lines emitted to stderr by the logger.')

=== Production Pattern Tests ===

Test 1 — Valid call:
  Result: $189.45

Test 2 — Schema violation (ticker too long):
  Result: Input validation failed: 'TOOLONGTICKER' is too long

Test 3 — Missing required field:
  Result: Input validation failed: 'drug_b' is a required property

Test 4 — Valid drug interaction call:
  Result: MAJOR interaction: increases bleeding risk.

Check above for JSON audit log lines emitted to stderr by the logger.


## Summary

### What MCP Is

MCP is an open JSON-RPC 2.0 protocol that standardises how AI agents discover and call tools. It has three roles: **Host** (your application), **Client** (one per server connection, handles transport), **Server** (exposes tools via `tools/list` and `tools/call`). Two transports: **stdio** for local tools, **SSE over HTTP** for remote/production services.

### The Structural Equivalence

A LangChain `@tool` and an MCP tool definition are the same thing expressed in two different syntaxes. Function name = `name`. Docstring = `description`. Typed parameters = `inputSchema`. Converting between them is mechanical and shown in Cell 5.

### What Was Built

**`enterprise_mcp_server.py` (Cell 4)** — a standalone MCP server exposing 5 tools (3 finance, 2 healthcare) over stdio transport. No LangChain dependency. Callable from any MCP client.

**`agent_mcp_server.py` (Cell 8)** — an MCP server that wraps entire LangChain ReAct agents as MCP tools. Callers send a natural-language question; the full multi-step agent runs internally and returns its final answer. This makes LangChain agents callable from Claude Desktop, GitHub Copilot Agent Mode, VS Code extensions, and any other MCP host — with zero changes to those hosts.

**MCP Client (Cell 6)** — connects to a server, calls `tools/list` to discover tools, then calls `tools/call` directly without an LLM. Shows the raw protocol in action.

**Tool loader (Cell 9)** — discovers tools from an MCP server and wraps them as LangChain `StructuredTool` objects. This is the bridge in the other direction: a LangChain agent can consume tools from any MCP server without hardcoding `@tool` definitions.

**Multi-server aggregator (Cell 11)** — connects to multiple MCP servers simultaneously, merges their tool lists, and routes calls to the originating server. This is the enterprise supervisor pattern: one agent, many independently deployed capability servers.

**Production patterns (Cell 12)** — JSON Schema validation before execution, retry wrapper for unreliable APIs, structured audit logging per tool call. These three patterns are the minimum required for a production MCP server.

### When to Use MCP vs LangChain `@tool` Directly

| Scenario | Recommendation |
|---|---|
| Single Python application, no sharing | `@tool` directly — simpler |
| Tools shared across multiple agent frameworks | MCP server |
| Tools accessible from Claude Desktop / Copilot | MCP server |
| Team-owned capability exposed to other teams | MCP server |
| Rapid prototyping in a notebook | `@tool` directly |
| Production microservice with auth + monitoring | MCP server over SSE |